## Cell 1 — Bronze: Clickstream (JSON from S3)
Files were uploaded by `upload_to_s3.py` to `s3://ecommerce-lakehouse-databricks/stream/`

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

from pyspark.sql.functions import (
    to_timestamp,
    current_timestamp,
    col,
    lit
)

# ── Paths ─────────────────────────────────────────────
CLICKSTREAM_SOURCE     = "s3://ecommerce-lakehouse-databricks/stream/"
CLICKSTREAM_CHECKPOINT = "s3://ecommerce-lakehouse-databricks/checkpoints/clickstream/"
CLICKSTREAM_TABLE      = "ecommerce.bronze.clickstream"

# ── Schema ────────────────────────────────────────────
clickstream_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("page_url", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("referrer", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", DoubleType(), True),
    StructField("amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("time_spent_sec", IntegerType(), True),
    StructField("timestamp", StringType(), True)
])

clickstream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option(
        "cloudFiles.schemaLocation",
        CLICKSTREAM_CHECKPOINT + "schema/"
    )
    .option("multiLine", "true")
    .schema(clickstream_schema)
    .load(CLICKSTREAM_SOURCE)

    .withColumn(
        "event_ts",
        to_timestamp(
            "timestamp",
            "yyyy-MM-dd'T'HH:mm:ss"
        )
    )

    .withColumn(
        "ingested_at",
        current_timestamp()
    )

    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )

    .withColumn(
        "source_system",
        lit("flask_clickstream_api")
    )
)

query = (
    clickstream_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        CLICKSTREAM_CHECKPOINT
    )
    .outputMode("append")

    # continuously check for files every 30 sec
    #.trigger(processingTime="30 seconds")
    .trigger(availableNow=True)
    .table(CLICKSTREAM_TABLE)
)

print("Streaming started...")
print(query.id)

In [0]:
spark.sql("""
SELECT
COUNT(DISTINCT source_file) files_loaded
FROM ecommerce.bronze.clickstream
""").show()

In [0]:
display(
spark.sql("""
SELECT source_file,
COUNT(*) rows_loaded
FROM ecommerce.bronze.clickstream
GROUP BY source_file
""")
)

In [0]:
display(
    spark.table("ecommerce.bronze.clickstream")
    .limit(10)
)

In [0]:
tables = spark.sql(
    "SHOW TABLES IN ecommerce.bronze"
).collect()

for t in tables:
    
    table_name = t.tableName
    
    print("\n" + "="*70)
    print(f"TABLE: ecommerce.bronze.{table_name}")
    print("="*70)
    
    df = spark.table(
        f"ecommerce.bronze.{table_name}"
    )
    
    for field in df.schema.fields:
        print(
            f"Column: {field.name:<25} "
            f"Datatype: {field.dataType} "
            f"Nullable: {field.nullable}"
        )